In [32]:
import os
import zipfile
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Как же повезло, что я такой умный. Я уже подготовил валидационный датасет, это еще давно было. А теперь я просто расформировал его на два и выполню задание. Структура тут такая:
 - деление на вда датасета и инициализация лоадара
 - заполнение функций из сопроводительного ноутбука
 - тестирование этих функций на тестах от DLS
 - финальный прогон подсчёта метрик

# Деление на датасеты

In [3]:
ZIP_VAL_PATH = '/content/drive/MyDrive/Дз Ноутбуки DLS/Первый поток/Проект 1/Dataset/aligned_val.zip'
EXTRACT_VAL_DIR = './aligned_val' # распакуем в локальную директорию сессии

print("📦 Распаковка валидационного датасета...")
with zipfile.ZipFile(ZIP_VAL_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_VAL_DIR)
print("✅ Распаковка завершена!")

📦 Распаковка валидационного датасета...
✅ Распаковка завершена!


In [4]:
ANNO_PATH = '/content/drive/MyDrive/Дз Ноутбуки DLS/Первый поток/Проект 1/CelebA/Anno/identity_CelebA.txt'

# Читаем файл разметки (разделитель — пробел)
df_anno = pd.read_csv(ANNO_PATH, sep=r'\s+', names=['image_name', 'label'], header=None)

# Получаем список файлов, которые реально распаковались в папку
# (Обычно внутри архива может быть еще одна подпапка, проверяем этот момент)
existing_images = set(os.listdir(EXTRACT_VAL_DIR))

# Если внутри оказалась еще одна папка (например, 'aligned_val/aligned_val'), корректируем путь
if len(existing_images) == 1 and os.path.isdir(os.path.join(EXTRACT_VAL_DIR, list(existing_images)[0])):
    EXTRACT_VAL_DIR = os.path.join(EXTRACT_VAL_DIR, list(existing_images)[0])
    existing_images = set(os.listdir(EXTRACT_VAL_DIR))

# Оставляем в датафрейме только те картинки, которые есть в нашей val-папке
df_val = df_anno[df_anno['image_name'].isin(existing_images)].reset_index(drop=True)

print(f" Всего найдено картинок в папке val: {len(df_val)}")
print(f" Уникальных людей в val: {df_val['label'].nunique()}")

 Всего найдено картинок в папке val: 300
 Уникальных людей в val: 50


In [5]:
def split_val_dataset_strict(df, query_id_count=20):
    # Фильтруем людей, у которых строго >= 2 фотографии для query
    class_counts = df['label'].value_counts()
    valid_query_ids = class_counts[class_counts >= 2].index.tolist()

    # Фиксируем seed для воспроизводимости тестов
    np.random.seed(42)

    # Выбираем ровно 20 человек для query
    query_ids = np.random.choice(valid_query_ids, size=query_id_count, replace=False)
    query_ids_set = set(query_ids)

    # Все остальные люди идут в distractors
    all_distinct_ids = df['label'].unique()
    distractor_ids = [idx for idx in all_distinct_ids if idx not in query_ids_set]

    # Берем топ-30 человек для distractors, как ты уточнил
    distractor_ids = distractor_ids[:30]
    distractor_ids_set = set(distractor_ids)

    # Собираем итоговые датафреймы
    df_query = df[df['label'].isin(query_ids_set)].reset_index(drop=True)
    df_distractors = df[df['label'].isin(distractor_ids_set)].reset_index(drop=True)

    # Словарь для positive/negative пар внутри query
    query_dict = df_query.groupby('label')['image_name'].apply(list).to_dict()

    return df_query, df_distractors, query_dict

In [7]:
df_query, df_distractors, query_dict = split_val_dataset_strict(df_val, query_id_count=20)

query_img_names = df_query['image_name'].tolist()
distractors_img_names = df_distractors['image_name'].tolist()

print(f"\n📊 Итоги разбиения:")
print(f"├── Query: {len(df_query)} фото, {df_query['label'].nunique()} уникальных людей")
print(f"└── Distractors: {len(df_distractors)} фото, {df_distractors['label'].nunique()} уникальных людей")


📊 Итоги разбиения:
├── Query: 120 фото, 20 уникальных людей
└── Distractors: 180 фото, 30 уникальных людей


In [12]:
shared_transforms = T.Compose([
    T.CenterCrop((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CelebAClassificationDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        """
        df: DataFrame со сплитом (содержит колонки 'image_name' и 'label')
        img_dir: Путь к конкретной распакованной папке с картинками (train или test)
        transform: Пайплайн трансформаций
        """
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_name'])

        # Открываем изображение и конвертируем в RGB
        image = Image.open(img_path).convert('RGB')
        label = int(row['label'])

        if self.transform:
            image = self.transform(image)

        return image, label


query_dataset = CelebAClassificationDataset(
    df=df_query,
    img_dir=EXTRACT_VAL_DIR,
    transform=shared_transforms
)

distractors_dataset = CelebAClassificationDataset(
    df=df_distractors,
    img_dir=EXTRACT_VAL_DIR,
    transform=shared_transforms
)

EVAL_BATCH_SIZE = 64

# shuffle=False ОБЯЗАТЕЛЕН, чтобы порядок эмбеддингов совпал с именами файлов [cite: 4]
query_loader = DataLoader(
    query_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

distractors_loader = DataLoader(
    distractors_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

# функции

In [13]:
def compute_embeddings(model, images_list):
    '''
    compute embeddings from the trained model for list of images.
    '''
    # Для инференса на реальном датасете используем наши созданные лоадеры,
    # так как делать поштучный инференс для сотен путей через PIL — это долго.
    # Метод определяет, к какому набору относятся имена файлов.
    import __main__

    # Пытаемся автоматически определить нужный loader
    if hasattr(__main__, 'query_img_names') and images_list == getattr(__main__, 'query_img_names'):
        loader = getattr(__main__, 'query_loader')
    elif hasattr(__main__, 'distractors_img_names') and images_list == getattr(__main__, 'distractors_img_names'):
        loader = getattr(__main__, 'distractors_loader')
    else:
        # Если это кастомный вызов или тест, у нас нет лоадера — выходим (в тестах функция не вызывается)
        return []

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    all_embeddings = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            # Извлекаем чистые 512-мерные эмбеддинги
            embeddings = model(images, return_embeddings=True)
            all_embeddings.append(embeddings.cpu().numpy())

    # Конкатенируем и возвращаем как список
    return np.vstack(all_embeddings).tolist()

In [14]:
def compute_cosine_query_pos(query_dict, query_img_names, query_embeddings):
    '''
    compute cosine similarities between positive pairs from query (stage 1)
    '''
    # Переводим в numpy и мапим имя файла -> его эмбеддинг
    embeddings_np = np.array(query_embeddings, dtype=np.float32)
    # Нормализуем для быстрого подсчета косинуса через скалярное произведение
    norms = np.linalg.norm(embeddings_np, axis=1, keepdims=True)
    embeddings_norm = embeddings_np / np.where(norms == 0, 1e-12, norms)

    name_to_emb = {name: emb for name, emb in zip(query_img_names, embeddings_norm)}
    similarities = []

    for cls_label, imgs in query_dict.items():
        if len(imgs) < 2:
            continue
        # Считаем все уникальные пары (i, j) одного человека
        for i in range(len(imgs)):
            for j in range(i + 1, len(imgs)):
                emb1 = name_to_emb[imgs[i]]
                emb2 = name_to_emb[imgs[j]]
                sim = float(np.dot(emb1, emb2))
                similarities.append(sim)

    return similarities

In [15]:
def compute_cosine_query_neg(query_dict, query_img_names, query_embeddings):
    '''
    compute cosine similarities between negative pairs from query (stage 2)
    '''
    embeddings_np = np.array(query_embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings_np, axis=1, keepdims=True)
    embeddings_norm = embeddings_np / np.where(norms == 0, 1e-12, norms)

    name_to_emb = {name: emb for name, emb in zip(query_img_names, embeddings_norm)}

    # Инвертируем словарь, чтобы знать, какому классу принадлежит каждая картинка
    img_to_class = {}
    for cls_label, imgs in query_dict.items():
        for img in imgs:
            img_to_class[img] = cls_label

    similarities = []

    # Сравниваем все картинки из query между собой
    for i in range(len(query_img_names)):
        for j in range(i + 1, len(query_img_names)):
            img1 = query_img_names[i]
            img2 = query_img_names[j]

            # Если классы разные — это негативная пара
            if img_to_class[img1] != img_to_class[img2]:
                emb1 = name_to_emb[img1]
                emb2 = name_to_emb[img2]
                sim = float(np.dot(emb1, emb2))
                similarities.append(sim)

    return similarities

In [16]:
def compute_cosine_query_distractors(query_embeddings, distractors_embeddings):
    '''
    compute cosine similarities between negative pairs from query and distractors (stage 3)
    '''
    q_emb = np.array(query_embeddings, dtype=np.float32)
    d_emb = np.array(distractors_embeddings, dtype=np.float32)

    # Нормализация матриц по строкам
    q_norms = np.linalg.norm(q_emb, axis=1, keepdims=True)
    q_norm = q_emb / np.where(q_norms == 0, 1e-12, q_norms)

    d_norms = np.linalg.norm(d_emb, axis=1, keepdims=True)
    d_norm = d_emb / np.where(d_norms == 0, 1e-12, d_norms)

    # Матричное умножение дает полную сетку сходств размера [len(query) x len(distractors)]
    sim_matrix = np.dot(q_norm, d_norm.T)

    # Разворачиваем в плоский одномерный Python-список, как требует шаблон
    return sim_matrix.flatten().tolist()

In [26]:
def compute_ir(cosine_query_pos, cosine_query_neg, cosine_query_distractors, fpr=0.1):
    '''
    compute identification rate using precomputer cosine similarities between pairs
    at given fpr
    '''
    # 1. Объединяем шаги 2 и 3, получая все false пары
    false_pairs = np.array(cosine_query_neg + cosine_query_distractors, dtype=np.float32)
    pos_pairs = np.array(cosine_query_pos, dtype=np.float32)

    total_false = len(false_pairs)

    # 2. Считаем N без округлений сторонними функциями, просто через int()
    N = int(total_false * fpr)

    # 3. Отсортируем все значения косинусных сходств false пар в порядке убывания
    sorted_false = np.sort(false_pairs)[::-1]

    # 4. Берем ровно N-ый элемент из отсортированного массива по правилам тестов
    threshold = float(sorted_false[N])

    # 5. Считаем количество positive пар, которые имеют сходство >= threshold
    true_positives = np.sum(pos_pairs >= threshold)
    tpr = float(true_positives / len(pos_pairs))

    return threshold, tpr

# Тесты от DLS

In [29]:
test_query_dict = {
    2876: ['1.jpg', '2.jpg', '3.jpg'],
    5674: ['5.jpg'],
    864:  ['9.jpg', '10.jpg'],
}
test_query_img_names = ['1.jpg', '2.jpg', '3.jpg', '5.jpg', '9.jpg', '10.jpg']
test_query_embeddings = [
                    [1.56, 6.45,  -7.68],
                    [-1.1 , 6.11,  -3.0],
                    [-0.06,-0.98,-1.29],
                    [8.56, 1.45,  1.11],
                    [0.7,  1.1,   -7.56],
                    [0.05, 0.9,   -2.56],
]

test_distractors_img_names = ['11.jpg', '12.jpg', '13.jpg', '14.jpg', '15.jpg']

test_distractors_embeddings = [
                    [0.12, -3.23, -5.55],
                    [-1,   -0.01, 1.22],
                    [0.06, -0.23, 1.34],
                    [-6.6, 1.45,  -1.45],
                    [0.89,  1.98, 1.45],
]

test_cosine_query_pos = compute_cosine_query_pos(test_query_dict, test_query_img_names,
                                            test_query_embeddings)
test_cosine_query_neg = compute_cosine_query_neg(test_query_dict, test_query_img_names,
                                            test_query_embeddings)
test_cosine_query_distractors = compute_cosine_query_distractors(test_query_embeddings,
                                                            test_distractors_embeddings)

In [30]:
true_cosine_query_pos = [0.8678237233650096, 0.21226104378511604,
                         -0.18355866977496182, 0.9787437979250561]
assert np.allclose(sorted(test_cosine_query_pos), sorted(true_cosine_query_pos)), \
      "A mistake in compute_cosine_query_pos function"

true_cosine_query_neg = [0.15963231223161822, 0.8507997093616965, 0.9272761484302097,
                         -0.0643994061127092, 0.5412660901220571, 0.701307100338029,
                         -0.2372575528216902, 0.6941032794522218, 0.549425446066643,
                         -0.011982733001947084, -0.0466679194884999]
assert np.allclose(sorted(test_cosine_query_neg), sorted(true_cosine_query_neg)), \
      "A mistake in compute_cosine_query_neg function"

true_cosine_query_distractors = [0.3371426578637511, -0.6866465610863652, -0.8456563512871669,
                                 0.14530087113136106, 0.11410510307646118, -0.07265097629002357,
                                 -0.24097699660707042,-0.5851992679925766, 0.4295494455718534,
                                 0.37604478596058194, 0.9909483738948858, -0.5881093317868022,
                                 -0.6829712976642919, 0.07546364489032083, -0.9130970963915521,
                                 -0.17463101988684684, -0.5229363015558941, 0.1399896725311533,
                                 -0.9258034013399499, 0.5295114163723346, 0.7811585442749943,
                                 -0.8208760031249596, -0.9905139680301821, 0.14969764653247228,
                                 -0.40749654525418444, 0.648660814944824, -0.7432584300096284,
                                 -0.9839696492435877, 0.2498741082804709, -0.2661183373780491]
assert np.allclose(sorted(test_cosine_query_distractors), sorted(true_cosine_query_distractors)), \
      "A mistake in compute_cosine_query_distractors function"

In [27]:
test_thr = []
test_tpr = []
for fpr in [0.5, 0.3, 0.1]:
  x, y = compute_ir(test_cosine_query_pos, test_cosine_query_neg,
                    test_cosine_query_distractors, fpr=fpr)
  test_thr.append(x)
  test_tpr.append(y)

In [28]:
true_thr = [-0.011982733001947084, 0.3371426578637511, 0.701307100338029]
assert np.allclose(np.array(test_thr), np.array(true_thr)), "A mistake in computing threshold"

true_tpr = [0.75, 0.5, 0.5]
assert np.allclose(np.array(test_tpr), np.array(true_tpr)), "A mistake in computing tpr"

Как видите все пройдено без ошибок, не с первого раза, но пройдено.

# Инициализация модели

Если вы смотрели мою работу до этого, то это то же самый код инициализации что и везде

In [33]:
class FaceReductionNet(nn.Module):
    def __init__(self, num_classes=700, embedding_dim=512):
        super(FaceReductionNet, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32), nn.Mish(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32), nn.Mish(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.05)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64), nn.Mish(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64), nn.Mish(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.1)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128), nn.Mish(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128), nn.Mish(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256), nn.Mish(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.3)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.embedding_layer = nn.Sequential(
            nn.Linear(256, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, return_embeddings=False):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        embeddings = self.embedding_layer(x)

        if return_embeddings:
            return embeddings
        return self.classifier(embeddings)

In [37]:
os.makedirs('weights', exist_ok=True)

!gdown --id 1ERcn8yOxfrEUyP80dAZM0dhEDlt5gi72 -O weights/arcface_best.pth

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_arcface = FaceReductionNet()

try:
    arcface_checkpoint = torch.load('weights/arcface_best.pth', map_location=device)
    # Извлекаем только веса бэкбона из чекпоинта ArcFace
    if isinstance(arcface_checkpoint, dict) and 'model_state_dict' in arcface_checkpoint:
        model_arcface.load_state_dict(arcface_checkpoint['model_state_dict'])
    else:
        model_arcface.load_state_dict(arcface_checkpoint)

    model_arcface = model_arcface.to(device)
    model_arcface.eval() # Обязательно выключаем Dropout2d
    print("[ОК] Веса FaceReductionNet (ArcFace) успешно загружены.")
except Exception as e:
    print(f"[ОШИБКА] Не удалось загрузить веса ArcFace: {e}")


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1ERcn8yOxfrEUyP80dAZM0dhEDlt5gi72
To: /content/weights/arcface_best.pth
100% 5.76M/5.76M [00:00<00:00, 183MB/s]
[ОК] Веса FaceReductionNet (ArcFace) успешно загружены.


# Главный тест

In [38]:
# 1. Извлекаем эмбеддинги для реального валидационного датасета
# Передаем именно model_arcface, которую мы только что инициализировали
print("🚀 Извлечение эмбеддингов для query сета...")
query_embeddings = compute_embeddings(model_arcface, query_img_names)

print("🚀 Извлечение эмбеддингов для distractors сета...")
distractors_embeddings = compute_embeddings(model_arcface, distractors_img_names)

# 2. Вычисляем косинусные сходства между всеми нужными парами
print("📊 Расчет косинусных сходств пар...")
cosine_query_pos = compute_cosine_query_pos(query_dict, query_img_names, query_embeddings)
cosine_query_neg = compute_cosine_query_neg(query_dict, query_img_names, query_embeddings)
cosine_query_distractors = compute_cosine_query_distractors(query_embeddings, distractors_embeddings)

print(f" Найдено положительных пар (Positive): {len(cosine_query_pos)}")
print(f" Найдено отрицательных пар внутри query: {len(cosine_query_neg)}")
print(f" Найдено пар query <-> distractors: {len(cosine_query_distractors)}")

# 3. Считаем итоговые метрики TPR@FPR для заданных преподавателями значений
print("\n🔥 ИТОГОВЫЕ МЕТРИКИ IDENTIFICATION RATE (TPR@FPR):")
print("=" * 50)

fpr_values = [0.5, 0.2, 0.1, 0.05]
for fpr in fpr_values:
    threshold, tpr = compute_ir(cosine_query_pos, cosine_query_neg, cosine_query_distractors, fpr=fpr)
    print(f"TPR @ FPR = {fpr:<4} | Порог (Threshold): {threshold:+.4f} | Метрика (TPR): {tpr * 100:.2f}%")

print("=" * 50)

🚀 Извлечение эмбеддингов для query сета...
🚀 Извлечение эмбеддингов для distractors сета...
📊 Расчет косинусных сходств пар...
 Найдено положительных пар (Positive): 300
 Найдено отрицательных пар внутри query: 6840
 Найдено пар query <-> distractors: 21600

🔥 ИТОГОВЫЕ МЕТРИКИ IDENTIFICATION RATE (TPR@FPR):
TPR @ FPR = 0.5  | Порог (Threshold): +0.0240 | Метрика (TPR): 98.33%
TPR @ FPR = 0.2  | Порог (Threshold): +0.1803 | Метрика (TPR): 93.67%
TPR @ FPR = 0.1  | Порог (Threshold): +0.2643 | Метрика (TPR): 87.00%
TPR @ FPR = 0.05 | Порог (Threshold): +0.3308 | Метрика (TPR): 79.67%


На самом деле, вроде как очень крутой результат, как для невероятно легковесной модели. Хотя учитывая что такие модели стоят уже даже на часах электронных, можно догадаться что в контексте этой задачи размер не особо влияет.